[Guide Index](README.md)


# 03b SPARQL-RL (SRL)

StarLayer provides support for SPARQL-RL (SRL), a general-purpose Datalog-style rules language published at [SPARQL 1.2 RL](https://www.w3.org/TR/sparql12-rl/). 



## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import StarLayerGraph, Namespace

EX = Namespace("http://example.org/")


## 1. The base graph

The following graph is used in the SRL examples that follow.

A small software-dependency graph — the same running example [the SRL spec's §6.6 worked example](https://www.w3.org/TR/sparql12-rl/#worked-example) uses: 
    
a `:frontend` depends on an `:app`, which depends on a `:db` that has a known vulnerability; a `:logger` component has no vulnerabilities at all. The rules below will propagate the vulnerability along the dependency chain.

In [2]:
base = StarLayerGraph()
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .  # no vulnerability - stays safe in section 4

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")

base.bind("ex", EX)

print("data graph")
print(base.serialize())

data graph
@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:app ex:dependsOn ex:db ;
    a ex:Component .

ex:db ex:hasVulnerability ex:vuln1 ;
    a ex:Component .

ex:frontend ex:dependsOn ex:app ;
    a ex:Component .

ex:logger a ex:Component .

ex:vuln1 ex:severity 9.1 .



## 2. Write and parse SRL rules

SRL rules follow a simple structure:  `RULE { head } WHERE { body }`

SRL rules work like a SPARQL `CONSTRUCT`: the `WHERE` pattern matches triples in the graph, and the head says which new triples to create. Two things are different. A rule can match triples that other rules created, so rules can build on each other. A rule can even build on its own output. Because of this, SRL runs the whole rule set repeatedly, adding the new triples each time, until a pass produces nothing new.

The two rules below show both. R1 creates `ex:exposedTo` triples from `ex:hasVulnerability`. R2 matches its own output: `ex:exposedTo` spreads through any number of `ex:dependsOn` hops.

`srl.parse_ruleset()` parses SRL text into a `srl.RuleSet`, a plain Python dataclass tree. `srl.ruleset_to_text()` renders the ruleset back to SRL text.

In [3]:
from starlayer.sparql import srl

rules_text = """
PREFIX ex: <http://example.org/>

# R1: a component is exposed to any vulnerability it has directly.
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }

# R2: exposure propagates along dependsOn (recursive - depends on itself).
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }
"""

ruleset = srl.parse_ruleset(rules_text)
print(f"{len(ruleset.rules)} rules parsed")
print(srl.ruleset_to_text(ruleset, namespace_manager=base.namespace_manager))


2 rules parsed
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }


## 3. Run inference

`ruleset.infer(base_graph)` evaluates the rule set and returns a *new* graph containing only the newly-inferred triples. It runs repeated passes until no new triples appear.

In this example the database has a vulnerability. R1 finds that the database is exposed. On the next pass R2 finds that the application, which depends on the database, is exposed. On the pass after that R2 finds that the front-end, which depends on the application, is exposed. A further pass produces nothing new, so evaluation stops.

In [4]:
# inferred is the set of new triples.
inferred = ruleset.infer(base)
inferred.bind("ex", EX)
print(inferred.serialize())


@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 .

ex:db ex:exposedTo ex:vuln1 .

ex:frontend ex:exposedTo ex:vuln1 .




## 4. `FILTER` and negation

Two more rules: R3 flags a component as critically exposed if any of its vulnerabilities has `severity >= 9.0` (`FILTER`); R4 marks a component safe if it's *not* critically exposed (`NOT { }`, negation-as-failure).

R4 genuinely depends on R3 having finished. SRL's stratification algorithm works this out from the rules' structure, so every component's exposure is fully propagated *before* R4 runs.

In [5]:

rules_text += """
# R3: severity >= 9.0 counts as a critical vulnerability.
RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

# R4: not critically exposed -> safe to deploy.
RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
inferred = ruleset.infer(base)
inferred.bind("ex", EX)
print()
print("inferred graph")
print(inferred.serialize(format="turtle"))



inferred graph
@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:logger ex:status ex:safeToDeploy .




## 5. Seeing the evaluation plan

Before running anything, SRL creates a plan that describes how the rules will be revalauted.  `ruleset.plan()` provide visibility to the plan: the **strata** in evaluation order, and the **dependency edges** between rules that the strata were worked out from. Rules are identified by their position in `ruleset.rules`, so R1 is index 0.

An edge `source -> target` means the `source` rule's body reads something the `target` rule's head can produce. It is **open** for an ordinary dependency, and **closed** when the source reads the target through `NOT { }` (or is a rule that must see its inputs complete). A closed edge forces the source into a *later* stratum than the target.

For the four rules from section 4, R4 has a closed edge to R3, so it waits for the stratum after R3:

In [6]:
plan = ruleset.plan()

for stratum in plan.strata:
    print(f"stratum {stratum.number}:", ", ".join(f"R{r.index + 1}" for r in stratum.general))

for edge in sorted(plan.edges, key=lambda e: (e.source, e.target)):
    print(f"R{edge.source + 1} -> R{edge.target + 1} ({edge.label})")

assert [[r.index for r in s.general] for s in plan.strata] == [[0, 1, 2], [3]]
assert {(e.source, e.target): e.label for e in plan.edges}[(3, 2)] == "closed"

stratum 0: R1, R2, R3
stratum 1: R4
R2 -> R1 (open)
R2 -> R2 (open)
R3 -> R1 (open)
R3 -> R2 (open)
R4 -> R3 (closed)


### Rule order doesn't matter

The plan comes from the rules' *structure*, not from the order they are written in. Below, R4 is written and listed before R3, and each rule is given an id so the plan can name it. R4 still lands in the later stratum, and `infer()` produces exactly the same result as section 4.

In [7]:
reordered_rules_text = """
PREFIX ex: <http://example.org/>

RULE ex:R1 { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE ex:R2 { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

# R4 listed before R3 on purpose - it has a closed dependency on R3 through NOT { }.
RULE ex:R4 { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }

RULE ex:R3 { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }
"""

reordered_ruleset = srl.parse_ruleset(reordered_rules_text)

for stratum in reordered_ruleset.plan().strata:
    print(f"stratum {stratum.number}:", ", ".join(base.qname(r.id) for r in stratum.general))

reordered_inferred = reordered_ruleset.infer(base)
print("same result regardless of rule order:", set(reordered_inferred) == set(inferred))
assert set(reordered_inferred) == set(inferred)

stratum 0: ex:R1, ex:R2, ex:R3
stratum 1: ex:R4
same result regardless of rule order: True


## 6. When no order exists

Some rule sets have no valid evaluation order. A rule that matches `NOT { }` of something it produces itself can never be settled: it would need its own output to be complete before it could run. SRL's §4.4.1 leaves this undefined, so StarLayer raises `StratificationError`, from `plan()` and from `infer()`.

This is also why a rule set cannot run forever. A rule that only reuses terms already in the graph can only ever produce a finite number of triples, so the repeated passes must stop. The only rules that could invent new values are those with an assignment (`SET(...)`) or a blank node in the head. Those run once per stratum, and a cycle through one is rejected in the same way.

In [8]:
from starlayer.sparql.srl_eval import StratificationError

unstratifiable = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:flagged ex:yes } WHERE { ?x a ex:Component . NOT { ?x ex:flagged ex:yes } }
""")

for attempt in (unstratifiable.plan, lambda: unstratifiable.infer(base)):
    try:
        attempt()
    except StratificationError as e:
        print("StratificationError:", e)

StratificationError: SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule set has no well-defined evaluation outcome per §4.4.1
StratificationError: SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule set has no well-defined evaluation outcome per §4.4.1


## 7. Asking a question with `query()`

`ruleset.query(base_graph, goal_pattern)` allows for a triple pattern query against the result of running rules against a data graph. Each match is a dictionary from variable to value. Here, which components are critically exposed?

In [9]:
from rdflib import Variable

goal = srl.TriplePattern(subject=Variable("x"), predicate=EX.status, object=EX.criticallyExposed)

for row in ruleset.query(base, goal):
    print(base.qname(row[Variable("x")]))

ex:db
ex:app
ex:frontend


## 8. Catching a malformed rule before evaluating it

SRL's own [§4.2 Well-formedness Conditions](https://www.w3.org/TR/sparql12-rl/#well-formedness-conditions) - every variable used in a rule's head must actually be bound by its own body - are checked automatically *inside* `srl.parse_ruleset()`. A rule set that violates this raises `SRLParseError` at parse time, rather than parsing and silently producing a wrong answer later. This is an easy mistake to make by hand, for example with a typo'd variable name.

In [10]:
try:
    broken = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?x ex:flaggedBy ?reviewer } WHERE { ?x ex:hasVulnerability ?v . }
    """)
except srl.SRLParseError as e:
    print("SRLParseError:", e)


SRLParseError: SRL: rule set is not well-formed per §4.2: unbound_head_variable (reviewer in rule head)


## 9. `DATA` blocks

A rule set can carry its own ground facts in a `DATA { }` block.  The facts travel with the rules and get combined with the base graph before the rules are run. 

A rule matches them exactly as it matches triples from the base graph, and they also appear in the output.

Here the rule set records which team owns two of the components, and one rule derives a `reportsTo` triple from each:

In [11]:
team_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . ex:app ex:team ex:web . }

RULE { ?x ex:reportsTo ?t } WHERE { ?x ex:team ?t }
""")

team_inferred = team_rules.infer(base)
team_inferred.bind("ex", EX)
print(team_inferred.serialize())

assert (EX.db, EX.team, EX.platform) in team_inferred      # the DATA facts are in the output
assert (EX.db, EX.reportsTo, EX.platform) in team_inferred # and the rule matched them

@prefix ex: <http://example.org/> .

ex:app ex:reportsTo ex:web ;
    ex:team ex:web .

ex:db ex:reportsTo ex:platform ;
    ex:team ex:platform .




### `WHERE DATA`: matching the original input graph

The name is a trap: `WHERE DATA` does **not** mean "match the `DATA { }` block". It means "match only the original input graph", the one you pass to `infer()`.

| A rule's `WHERE` matches... | plain `WHERE { }` | `WHERE DATA { }` |
|---|---|---|
| the base graph passed to `infer()` | yes | yes |
| the rule set's own `DATA { }` facts | yes | **no** |
| triples inferred by the rules so far | yes | **no** |

Use it for a rule that should look only at the data you supplied, never at the rule set's own facts or at other rules' conclusions. This is how the spec's evaluation algorithm (§6.5) defines it.

The same `DATA` fact is used below by three rules:
- the first uses a plain `WHERE`, so it matches the `DATA` fact;
- the second uses `WHERE DATA` for the same pattern, so it matches nothing, because `ex:team` exists only in the `DATA` block, not in the base graph;
- the third uses `WHERE DATA` for `ex:hasVulnerability`, which *is* in the base graph, so it matches.

In [12]:
scope_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . }

RULE { ?x ex:seenInData ?t }  WHERE      { ?x ex:team ?t }
RULE { ?x ex:seenInBase ?t }  WHERE DATA { ?x ex:team ?t }
RULE { ?x ex:flagged ex:yes } WHERE DATA { ?x ex:hasVulnerability ?v }
""")

scope_inferred = scope_rules.infer(base)
scope_inferred.bind("ex", EX)
print(scope_inferred.serialize())

assert (EX.db, EX.seenInData, EX.platform) in scope_inferred       # ordinary WHERE sees the DATA fact
assert not list(scope_inferred.triples((None, EX.seenInBase, None)))  # WHERE DATA does not
assert (EX.db, EX.flagged, EX.yes) in scope_inferred               # WHERE DATA sees the base graph

@prefix ex: <http://example.org/> .

ex:db ex:flagged ex:yes ;
    ex:seenInData ex:platform ;
    ex:team ex:platform .




### `NOT DATA`: filling in a default

`DATA` can also be used on a single `NOT`. The spec's own use is a **default value**: derive a value only when the base graph does not already state one. The rule below gives every vulnerable component a review priority unless the base graph already has one.

The rule negates the very predicate it produces (`ex:reviewPriority`). With a plain `NOT { }` that is an unstratifiable cycle, because the rule would need its own output to be finished before it could run. `NOT DATA { }` looks only at the base graph, so there is no cycle, and the rest of the rule body can still match inferred triples.

In [13]:
default_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:reviewPriority ex:high }
WHERE { ?x ex:hasVulnerability ?v . NOT DATA { ?x ex:reviewPriority ?existing } }
""")

# the base graph has no priority for ex:db, so the default is derived
print("no priority in the base graph:")
default_inferred = default_rules.infer(base)
default_inferred.bind("ex", EX)
print(default_inferred.serialize())

# a base graph that already states a priority is left alone
with_priority = StarLayerGraph()
with_priority += base
with_priority.add((EX.db, EX.reviewPriority, EX.low))
print("priority already in the base graph:", len(default_rules.infer(with_priority)), "triples inferred")

assert (EX.db, EX.reviewPriority, EX.high) in default_rules.infer(base)
assert len(default_rules.infer(with_priority)) == 0

# the same rule with a plain NOT cannot be ordered
plain_not = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:reviewPriority ex:high }
WHERE { ?x ex:hasVulnerability ?v . NOT { ?x ex:reviewPriority ?existing } }
""")
try:
    plain_not.plan()
except StratificationError as e:
    print("plain NOT:", type(e).__name__)

no priority in the base graph:
@prefix ex: <http://example.org/> .

ex:db ex:reviewPriority ex:high .


priority already in the base graph: 0 triples inferred
plain NOT: StratificationError


## 10. SRL as a syntax tree

A parsed rule set can also be written as an RDF syntax tree; see [SPARQL-RL rules as RDF](05c-sparql-rules-as-rdf.ipynb).

## 11. Not yet covered

These parts of SPARQL-RL are not in StarLayer yet, or not finished in the specification. This guide will gain a section for each when they are:

- **Aggregation** (`GROUP BY`, `COUNT`, `COUNT DISTINCT`, `MIN`, `MAX`, `SUM`). The draft describes an aggregation rule element, but it is still an open issue (Issue 840) with no grammar yet. Until then an aggregate inside `FILTER` or `SET` is rejected when the rule set is parsed.
- **Triple terms in rules.** Rule patterns and templates that mention a triple term (RDF 1.2). The rule set model already allows them, but the text syntax does not yet.
- **`IMPORTS`.** Rule sets that import other rule sets. The spec makes this optional, and `infer()` rejects it with a clear error.
- **Translating a rule set to SPARQL.** A placeholder appendix in the draft (Issue 1303).
- **Choosing among valid evaluation orders.** A rule set can have more than one valid stratification; `plan()` shows the one StarLayer uses.